In [148]:
import numpy as np
import pandas as pd
def data_preprocessing(path: str):
    data = pd.read_csv(path)
    data.head()
    #Title information
    data["Surname"] = data["Name"].str.split(",").str[0]
    data["Title"] = data["Name"].str.split(",").str[1].str.split(".").str[0].str.strip()
    data[data["Title"] == "Master"].value_counts()


    #FamilrySize

    data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
    data["IsAlone"] = (data["FamilySize"] == 1).astype(int)
    data[["FamilySize", "IsAlone"]].head()


    # Fare
    data["Fare"].describe()
    fare= np.log1p(data["Fare"])
    data["FareFloat"] = fare.astype(float)
    data = data.drop(["Fare"], axis=1)


    #Embarked
    mode = data["Embarked"].mode()[0]
    data["Embarked"] = data["Embarked"].fillna(mode)

    #Age
    data["Age_Missing"] = data["Age"].isnull().astype(int) 
    temp = data
    temp["Age"] = temp.groupby("Title")["Age"].transform(lambda x: x.fillna(x.median()))
    temp[temp["Age"].isnull()]
    data = temp

    #cabin

    temp = data
    temp["Cabin"] = temp["Cabin"].fillna("M").apply(lambda x: x[0])
    data= temp

    #ticket
    data["TicketCode"]= data["Ticket"].str.split(" ").str[0].transform(lambda x: "Unknown" if x.isdigit() else x)
    data["TicketNumber"]= data["Ticket"].str.split(" ").str[-1]

    data["TicketNumber"] = pd.to_numeric(data["TicketNumber"], errors="coerce")
    train_data = data.drop([ "Name", "Ticket"], axis=1)
    train_data = data.drop(["Survived"], axis=1) if "Survived" in data.columns else train_data
    train_label = data["Survived"] if "Survived" in data.columns else None
    
    return train_data, train_label

train_data, train_label = data_preprocessing("train.csv")
train_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Cabin,Embarked,Surname,Title,FamilySize,IsAlone,FareFloat,Age_Missing,TicketCode,TicketNumber
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,M,S,Braund,Mr,2,0,2.110213,0,A/5,21171.0
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,C,C,Cumings,Mrs,2,0,4.280593,0,PC,17599.0
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,M,S,Heikkinen,Miss,1,1,2.188856,0,STON/O2.,3101282.0
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,C,S,Futrelle,Mrs,2,0,3.990834,0,Unknown,113803.0
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,M,S,Allen,Mr,1,1,2.202765,0,Unknown,373450.0


In [142]:
cat_feature= ["Surname","Title", "Cabin", "Embarked", "TicketCode", "Sex"]


n_splits = 5
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
if train_label is not None:
    fold_accuracies = []
    for fold, (train_index, val_index) in enumerate(skf.split(train_data, train_label)):
        X_train, X_val = train_data.iloc[train_index], train_data.iloc[val_index]
        y_train, y_val = train_label.iloc[train_index], train_label.iloc[val_index]
        cat = CatBoostClassifier(
            iterations=1000,
            depth=8,
            learning_rate=0.03,
            loss_function="Logloss",
            eval_metric="Accuracy",
            random_seed=42,
            verbose=100,
            task_type="GPU"
        )
        cat.fit(X_train, y_train, cat_features=cat_feature, eval_set=(X_val, y_val), verbose=50)
        preds = cat.predict(X_val)
        acc = accuracy_score(y_val, preds)
        fold_accuracies.append(acc)
        print(f"Fold {fold + 1} Accuracy: {acc:.4f}")
    print(f"Average Accuracy across {n_splits} folds: {np.mean(fold_accuracies):.4f}")

0:	learn: 0.8441011	test: 0.8212291	best: 0.8212291 (0)	total: 109ms	remaining: 1m 49s
50:	learn: 0.8455056	test: 0.8491620	best: 0.8547486 (10)	total: 4.21s	remaining: 1m 18s
100:	learn: 0.8764045	test: 0.8491620	best: 0.8547486 (10)	total: 8.58s	remaining: 1m 16s
150:	learn: 0.9030899	test: 0.8547486	best: 0.8603352 (109)	total: 13s	remaining: 1m 12s
200:	learn: 0.9143258	test: 0.8659218	best: 0.8659218 (199)	total: 17.3s	remaining: 1m 8s
250:	learn: 0.9185393	test: 0.8603352	best: 0.8659218 (199)	total: 21.6s	remaining: 1m 4s
300:	learn: 0.9255618	test: 0.8603352	best: 0.8659218 (199)	total: 26.2s	remaining: 1m
350:	learn: 0.9339888	test: 0.8659218	best: 0.8659218 (199)	total: 31s	remaining: 57.3s
400:	learn: 0.9424157	test: 0.8659218	best: 0.8715084 (377)	total: 36s	remaining: 53.8s
450:	learn: 0.9522472	test: 0.8659218	best: 0.8715084 (377)	total: 40.7s	remaining: 49.5s
500:	learn: 0.9578652	test: 0.8715084	best: 0.8715084 (377)	total: 45.4s	remaining: 45.2s
550:	learn: 0.9592697	

In [149]:

test_data, _ = data_preprocessing("test.csv")
predictions = cat.predict(test_data)
submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": predictions
})
submission.to_csv("submission1.csv", index=False)

In [156]:
#boosting
cat_feature= ["Surname","Title", "Cabin", "Embarked", "TicketCode", "Sex"]
train_data, train_label = data_preprocessing("train.csv")
train_data=train_data.drop(["PassengerId","Name","Ticket"], axis=1)
X_encoded = train_data.copy()

X_encoded[cat_feature] = X_encoded[cat_feature].astype(str)
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier
encoder = OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value=-1)
X_encoded[cat_feature] = encoder.fit_transform(X_encoded[cat_feature])
cat_indices = [X_encoded.columns.get_loc(col) for col in cat_feature]
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_encoded, train_label, test_size=0.2, random_state=42, stratify=train_label)

model_cat = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="Accuracy", 
    random_seed=42,
    verbose=100,
    task_type="GPU",
    early_stopping_rounds=50
)
model_cat.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=100)
model_lgbm = LGBMClassifier(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.03,
    random_state=42,
    verbose=100,
    device="gpu",
    metric="binary_logloss",
)
import lightgbm as lgb
lgbm_early_stopping = lgb.early_stopping(stopping_rounds=50, verbose=False)
model_lgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)],callbacks=[lgbm_early_stopping])

model_xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.03,
    random_state=42,
    verbosity=1,
    eval_metric="logloss"   ,
    early_stopping_rounds=50
)                   
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)                                                         

probs_cat = model_cat.predict_proba(X_val)[:, 1]
probs_lgbm = model_lgbm.predict_proba(X_val)[:, 1] #type: ignore
probs_xgb = model_xgb.predict_proba(X_val)[:, 1]

final_probs = (probs_cat + probs_lgbm + probs_xgb) / 3
final_predictions = (final_probs >= 0.5).astype(int)
from sklearn.metrics import accuracy_score
ensemble_accuracy = accuracy_score(y_val, final_predictions)
print(f"Ensemble Accuracy: {ensemble_accuracy:.4f}")
                                             



        



0:	learn: 0.8370787	test: 0.7932961	best: 0.7932961 (0)	total: 37.3ms	remaining: 37.3s
100:	learn: 0.8960674	test: 0.8156425	best: 0.8268156 (97)	total: 4.65s	remaining: 41.4s
bestTest = 0.8268156425
bestIteration = 97
Shrink model to first 98 iterations.
[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.761236
[LightGBM] [Info] Total Bins 739
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 15
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Ti Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 9 dense feature groups (0.01 MB) transferred to GPU in 0.000813 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> 

In [ ]:
test_data, _ = data_preprocessing("test.csv")
passenger_id = test_data["PassengerId"].copy()
X_encoded = test_data.drop(["PassengerId"], axis=1).copy()
X_encoded[cat_feature] = X_encoded[cat_feature].astype(str)
X_encoded[cat_feature] = encoder.transform(X_encoded[cat_feature])
probs_cat = model_cat.predict_proba(X_encoded)[:, 1]
probs_lgbm = model_lgbm.predict_proba(X_encoded)[:, 1]  # type: ignore
probs_xgb = model_xgb.predict_proba(X_encoded)[:, 1]
final_probs = (probs_cat + probs_lgbm + probs_xgb) / 3
final_predictions = (final_probs >= 0.5).astype(int)
submission = pd.DataFrame({
    "PassengerId": passenger_id,
    "Survived": final_predictions
})
submission.to_csv("submission2.csv", index=False)

CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=1]="male": Cannot convert 'male' to float